# T4 · AdamW, step by step

**Tiny Transformer, Unit 2, module T4.** The way back gives every one of the 9,417 learned numbers a gradient. AdamW, the
run's optimizer, turns each into a step. Here you take the run's step 1 by hand on every number, follow one number's two averages,
replay the run's first 500 steps from its seed, work out weight decay's pull, and replay the first 1,000 steps without the
warm-up.

[Open in Google Colab](https://colab.research.google.com/github/gkiri/tiny_repo/blob/main/tiny-transformer/notebooks/T4-training-optimizer.ipynb) (execution uses Google's hosted runtime; no graphics card needed). On your own computer:
`pip install -r requirements.txt`, then `jupyter notebook T4-training-optimizer.ipynb`.

**Computed here:** step 1 by hand and by PyTorch, the first 500 steps, and the first 1,000 twice. **Checked against the run:** its
recorded first batch, its log, and the course's numbers, which the notebook asserts.

Laid out like the page: two sections, five chapters, and under each chapter its page's three steps (Picture it, Watch it in our
run, Read and run the code), then the code to run and a "your turn".

In [1]:
#@title Setup: run me first (one click) { display-mode: "form" }
# In Google Colab this fetches the course's labs (notebooks, data and the trained model) and moves into this notebook's
# folder, so every file is where the cells below look for it. On your own computer, inside a copy of the labs, it does nothing.
import os, subprocess, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/tiny_repo"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/gkiri/tiny_repo", "/content/tiny_repo"], check=True)
    os.chdir("/content/tiny_repo/tiny-transformer/notebooks")
print("ready:", os.path.basename(os.getcwd()))

ready: notebooks


In [2]:
import base64, hashlib, json, math, os, urllib.request
import numpy as np
import torch

torch.set_num_threads(1)     # a model this small runs fastest on one CPU thread
print(torch.__version__)

2.10.0+cpu


**The text, the model and this module's code.** Tiny Shakespeare, pinned by its sha256; then the model's file, `model.py`, copied
in whole; then Unit 2's teaching functions, `code/training_steps.py`, copied in whole: T4's are the last six.

In [3]:
URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
SHA256 = "86c4e6aa9db7c042ec79f339dcb96d42b0075e16b8fc2e86bf0ca57e2dc565ed"
here = os.path.abspath('')
candidates = [os.path.join(here, '..', 'data', 'tinyshakespeare.txt'),                                   # in the labs
              os.path.join(here, '..', '..', '..', '..', 'training', 'prompter', 'data', 'tinyshakespeare.txt')]   # in the course
found = [p for p in candidates if os.path.exists(p)]
raw = open(found[0], 'rb').read() if found else urllib.request.urlopen(URL, timeout=60).read()
assert hashlib.sha256(raw).hexdigest() == SHA256, "not the pinned Tiny Shakespeare"
text = raw.decode('utf-8')
print(f"{len(text):,} characters")

1,115,394 characters


In [4]:
"""Our tiny transformer: 9,417 learned numbers that predict the next character of Shakespeare.

It reads up to 32 characters of text and gives a probability to each of the 65 characters that could come next. Writing is
that prediction in a loop: pick a character, append it, slide the 32-character window, and predict again. (The class is
called Prompter for the course's file names; learners meet it as "our tiny transformer".)

The architecture is the "10k" configuration of maxpolaczuk/tiny-character-transformer (a 1-block, 2-head character GPT on Tiny
Shakespeare). The code here is written fresh for the course (that repository has no licence, so none of its code or weights
are copied): every line is one the course teaches.

    characters ─► token table (65 × 28) + position table (32 × 28)        lookup:      what, and where
               ─► one block: LayerNorm ─► 2-head causal attention ─► +    look back:   which earlier letters matter
                             LayerNorm ─► feed-forward 28 → 56 → 28 ─► +  think:       transform each row on its own
               ─► final LayerNorm ─► readout with the token table again   score all 65 characters
               ─► softmax                                                  probabilities for the next character

Parameters: token table 1,820 · positions 896 · block 6,580 (two LayerNorms 112, q|k|v 2,436, attention out 812,
feed-forward 1,624 + 1,596) · final LayerNorm 56 · readout bias 65 = 9,417.
"""

import math
from dataclasses import asdict, dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass(frozen=True)
class Config:
    vocab_size: int = 65    # every distinct character in Tiny Shakespeare
    block_size: int = 32    # how many characters the model can read at once (its context)
    d_model: int = 28       # how many numbers describe one character in one place (a row)
    n_head: int = 2         # attention heads, each looking with 28 / 2 = 14 numbers
    d_ff: int = 56          # the feed-forward network's hidden width (2 × d_model)
    n_layer: int = 1        # transformer blocks

    def to_dict(self):
        return asdict(self)


class Tokenizer:
    """One id per character, in sorted order: '\\n' is 0, ' ' is 1, … 'z' is 64."""

    def __init__(self, text: str):
        self.chars = sorted(set(text))
        self.stoi = {c: i for i, c in enumerate(self.chars)}

    def encode(self, s: str) -> list[int]:
        return [self.stoi[c] for c in s]

    def decode(self, ids) -> str:
        return "".join(self.chars[int(i)] for i in ids)


class CausalSelfAttention(nn.Module):
    """Each row asks a question (q), every earlier row offers a label (k) and a message (v). A row reads the messages of the
    rows whose labels match its question best, never a row to its right."""

    def __init__(self, cfg: Config):
        super().__init__()
        self.n_head, self.head_dim = cfg.n_head, cfg.d_model // cfg.n_head
        self.qkv = nn.Linear(cfg.d_model, 3 * cfg.d_model)     # q, k and v for both heads in one matrix
        self.proj = nn.Linear(cfg.d_model, cfg.d_model)        # mix the heads' results back into one row
        mask = torch.tril(torch.ones(cfg.block_size, cfg.block_size))
        self.register_buffer("mask", mask.view(1, 1, cfg.block_size, cfg.block_size), persistent=False)

    def forward(self, x, keep=None):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q, k, v = (t.view(B, T, self.n_head, self.head_dim).transpose(1, 2) for t in (q, k, v))
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)          # how well each question matches each label
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))   # no peeking at the future
        att = F.softmax(att, dim=-1)                                          # shares that add up to 1
        if keep is not None:
            keep["attn"] = att.detach()
        y = (att @ v).transpose(1, 2).contiguous().view(B, T, C)            # each row: a weighted mix of messages
        return self.proj(y)


class Block(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ff = nn.Sequential(nn.Linear(cfg.d_model, cfg.d_ff), nn.GELU(), nn.Linear(cfg.d_ff, cfg.d_model))

    def forward(self, x, keep=None):
        x = x + self.attn(self.ln1(x), keep)    # look back, and add what was found to the row
        x = x + self.ff(self.ln2(x))            # think about each row on its own, and add that too
        return x


class Prompter(nn.Module):
    def __init__(self, cfg: Config = Config()):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)    # one learned row per character
        self.pos_emb = nn.Embedding(cfg.block_size, cfg.d_model)    # one learned row per place 0 … 31
        self.blocks = nn.Sequential(*[Block(cfg) for _ in range(cfg.n_layer)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head_bias = nn.Parameter(torch.zeros(cfg.vocab_size))  # the readout reuses tok_emb (tied); only its bias is new
        self.apply(self._init)

    @staticmethod
    def _init(m):
        # small random numbers (σ = 0.02), so the untrained model's guesses are nearly even: its first loss is about
        # ln 65 = 4.17, the loss of a uniform guess over 65 characters
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
        if isinstance(m, nn.Linear) and m.bias is not None:
            nn.init.zeros_(m.bias)

    def forward(self, idx, targets=None, keep=None):
        T = idx.size(1)
        x = self.tok_emb(idx) + self.pos_emb(torch.arange(T, device=idx.device))   # what + where
        for b in self.blocks:
            x = b(x, keep)
        x = self.ln_f(x)
        logits = x @ self.tok_emb.weight.T + self.head_bias                         # a score for every character
        loss = None if targets is None else F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, n, temperature=1.0, generator=None):
        for _ in range(n):
            logits, _ = self(idx[:, -self.cfg.block_size:])
            probs = F.softmax(logits[:, -1, :] / temperature, dim=-1)
            nxt = torch.multinomial(probs, 1, generator=generator)
            idx = torch.cat([idx, nxt], dim=1)
        return idx


def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())

In [5]:
#@title Unit 2's teaching functions: code/training_steps.py, copied whole (run me) { display-mode: "form" }
"""Unit 2 · Inside the learning loop: the recorded run's training, station by station, done explicitly.

The teaching functions of Unit 2 (each T notebook carries this file whole, and the chapter pages light its lines; the first
line is a Colab form title, so the notebooks fold this cell and no chapter's question is answered by its code early). They sit
beside M6's code/training_loop.py, with its names (get_batch, loss_of, train_step, every_window, average_loss), and repeat the
recorded run's own code, training/prompter/train.py: test_training_steps.py checks them against it and against the run's
numbers (training/prompter/unit2.py's export, prompter-unit2.json).
"""
import math
import os

import torch


# ---------------------------------------------------------------------------------------------------- T1 · picking the batches

def check_starts(n_train, n_val, batches=100, B=64, T=32, seed=2024):
    """The checks' fixed windows, as train.py draws them: 100 batches of starts in the training text, then 100 in the held-back
    text."""
    g = torch.Generator().manual_seed(seed)                       # the checks' generator
    train = torch.stack([torch.randint(n_train - T - 1, (B,), generator=g) for _ in range(batches)])   # [batches×B]
    val = torch.stack([torch.randint(n_val - T - 1, (B,), generator=g) for _ in range(batches)])       # [batches×B]
    return train, val


def run_starts(n, steps=100_000, B=64, T=32, seed=1337, make_model=None):
    """Every step's B window starts, as the recorded run drew them (train.py): the main random stream, started by the seed,
    first fills the model's random starting numbers (make_model builds the model), then draws B starts a step."""
    torch.manual_seed(seed)                                       # the main stream: starting numbers, then every batch
    if make_model is not None:
        make_model()                                              # first: the model's random starting numbers
    return torch.stack([torch.randint(n - T - 1, (B,)) for _ in range(steps)])   # [steps×B]  then B starts every step


def times_drawn(starts, n, T=32):
    """How many times each possible start was drawn. randint(n - T - 1) never draws its limit: starts run 0 … n - T - 2."""
    return torch.bincount(starts.flatten(), minlength=n - T - 1)  # [n−T−1]   one count per possible start


def times_asked(starts, n, T=32):
    """How many times each character of the text was a right answer: a window from start i asks about i + 1 … i + T."""
    drawn = times_drawn(starts, n, T)                             # [n−T−1]
    change = torch.zeros(n + 1, dtype=torch.long)
    change[1:n - T] += drawn                                      # a window from i starts asking at i + 1 …
    change[T + 1:n] -= drawn                                      # … and stops after i + T
    return change.cumsum(0)[:n]                                   # [n]       one count per character


def times_through(n, steps=100_000, B=64, T=32):
    """How many times over the run reads its text: every step asks B × T questions, one per right answer."""
    return steps * B * T / n                                      # answers ÷ characters


# ---------------------------------------------------------------------------------------------------- T2 · the loss

def all_at_once(model, x, y):
    """A batch's right answers all at once: their probabilities multiplied, in the model's 32-bit numbers, and their costs,
    minus each one's log, added and averaged (the loss)."""
    probs = torch.softmax(model(x)[0], dim=-1)                    # [B×T×65]  probabilities, as M6's loss_of makes them
    p = probs.gather(-1, y.unsqueeze(-1)).squeeze(-1)             # [B×T]     each right answer's probability
    product = p.prod()                                            # []        all of them together: too small for 32 bits, 0.0
    total = -p.log().sum()                                        # []        their costs, added
    return product, total, total / p.numel()                      # the product, the sum, and the average: the loss


def loss_from_scores(model, x, y):
    """The loss as model.py's F.cross_entropy works it out, straight from the scores: for each answer, the log of the total of
    e to every score, minus the right answer's own score. No probability is ever made."""
    scores = model(x)[0]                                          # [B×T×65]  a score for every character
    right = scores.gather(-1, y.unsqueeze(-1)).squeeze(-1)        # [B×T]     each right answer's score
    return (scores.logsumexp(-1) - right).mean()                  # []        minus the log of each probability, averaged


def answer_costs(model, ids, T=32, per=256):
    """Every right answer's probability over a whole text, and whether it was the top pick, read as the final score reads it
    (train.py full_loss): windows side by side, every character scored once but the first and the last few, which fill no
    window."""
    n = len(range(0, len(ids) - T - 1, T))                        # how many whole windows fit
    x, y = ids[:n * T].view(n, T), ids[1:n * T + 1].view(n, T)    # [n×T] each: the inputs, and the targets moved one along
    p, top = [], []
    with torch.no_grad():
        for xb, yb in zip(x.split(per), y.split(per)):
            probs = torch.softmax(model(xb)[0], dim=-1)           # [256×T×65]
            p.append(probs.gather(-1, yb.unsqueeze(-1)).squeeze(-1))   # [256×T]  the right answer's probability
            top.append(probs.argmax(-1) == yb)                    # [256×T]  was it the top pick?
    return torch.cat(p).flatten(), torch.cat(top).flatten()       # [n·T] each


def share_of_loss(p, below=0.01):
    """The answers given a probability under `below`: their share of all the answers, and their share of the loss."""
    cost = -p.log()                                               # each answer's cost
    rare = p < below
    return rare.float().mean().item(), (cost[rare].sum() / cost.sum()).item()


def sure_or_right(p, top):
    """Accuracy, the share of answers whose top pick is right; and the share right but unsure: top pick, yet under one half."""
    return top.float().mean().item(), (top & (p < 0.5)).float().mean().item()


def choices(loss):
    """A loss read backwards: an even guess among n characters costs ln n on every answer, so n = e to the loss."""
    return math.exp(loss)


# ---------------------------------------------------------------------------------------------------- T3 · the way back

def bias_gradient(probs, y):
    """Every readout bias's gradient, by its rule. A bias adds to one character's score at every place, so its gradient is that
    character's probability averaged over the places, minus its share of the right answers."""
    V = probs.size(-1)
    predicted = probs.reshape(-1, V).mean(0)                      # [65]  each character's average probability
    right = torch.bincount(y.flatten(), minlength=V).float() / y.numel()   # [65]  its share of the right answers
    return predicted - right                                      # [65]  average probability, minus share of right answers


def nudge(model, x, y, param, i, h=1e-3):
    """One number's gradient without backpropagation: nudge it up by h and run the batch, then down by h and run it again, and
    take the slope through the two losses. The number is put back as it was."""
    flat = param.data.view(-1)
    old = flat[i].item()
    with torch.no_grad():
        flat[i] = old + h
        up = model(x, y)[1].item()                                # the loss, nudged up
        flat[i] = old - h
        down = model(x, y)[1].item()                              # the loss, nudged down
        flat[i] = old                                             # put back
    return (up - down) / (2 * h)


def grad_twice(model, x, y, param):
    """backward fills every learned number's .grad in one pass, and moves none of them. Run it again without clearing, and the
    gradients add up: why train.py clears them first, with opt.zero_grad()."""
    model.zero_grad()
    model(x, y)[1].backward()                                     # every .grad filled; no number moved
    once = param.grad.clone()
    model(x, y)[1].backward()                                     # again, without clearing
    return once, param.grad.clone()                               # the second: twice the first


def backward_sizes(model, x, y):
    """The gradient on its way back through model.py's pass, part by part: its size (all its numbers measured together) at the
    scores, at the block's top, into each add's branch, below each add, and at the rows the tables made."""
    blk = model.blocks[0]
    rows = model.tok_emb(x) + model.pos_emb(torch.arange(x.size(1)))   # [B×T×28]  token row + position row
    att = blk.attn(blk.ln1(rows))                                 # the attention's branch
    mid = rows + att                                              # its add
    ff = blk.ff(blk.ln2(mid))                                     # the feed-forward's branch
    top = mid + ff                                                # its add: the block's top
    scores = model.ln_f(top) @ model.tok_emb.weight.T + model.head_bias   # [B×T×65]  the readout
    loss = torch.nn.functional.cross_entropy(scores.view(-1, scores.size(-1)), y.view(-1))
    parts = {"scores": scores, "top": top, "ff_branch": ff, "meet": mid, "attn_branch": att, "rows": rows}
    grads = torch.autograd.grad(loss, list(parts.values()))       # every part's gradient, in one pass back
    return {name: g.norm().item() for name, g in zip(parts, grads)}, dict(zip(parts, grads))


def table_gradient_by_use(model, x, y):
    """The token table's gradient, one job at a time: read in at the start (each input's entry), and reused at the readout (scoring
    every character). Holding one use still (detach) leaves the other's push; the two add up to the whole gradient."""
    W = model.tok_emb.weight                                      # [65×28]

    def push(use):
        rows = (W if use == "in" else W.detach())[x] + model.pos_emb(torch.arange(x.size(1)))
        for b in model.blocks:
            rows = b(rows)
        scores = model.ln_f(rows) @ (W if use == "out" else W.detach()).T + model.head_bias
        loss = torch.nn.functional.cross_entropy(scores.view(-1, scores.size(-1)), y.view(-1))
        return torch.autograd.grad(loss, W)[0]                    # [65×28]

    return push("in"), push("out")


# ---------------------------------------------------------------------------------------------------- T4 · AdamW

def adamw_step(p, g, m, v, t, lr, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.01):
    """One AdamW step for any tensor of numbers, as torch.optim.AdamW takes it on PyTorch's defaults (train.py passes only the
    learning rate). m and v are the running averages of the pushes and of their squares, both 0 before step 1; t counts the
    steps from 1. Returns the moved numbers and the two averages."""
    b1, b2 = betas
    p = p * (1 - lr * weight_decay)                               # the W: weight decay, a pull toward 0 (T4.4)
    m = b1 * m + (1 - b1) * g                                     # the pushes, averaged with their signs (T4.3)
    v = b2 * v + (1 - b2) * g * g                                 # their sizes, squared and averaged
    m_hat = m / (1 - b1 ** t)                                     # both start at 0: the corrections (T4.2)
    v_hat = v / (1 - b2 ** t)
    step = m_hat / (v_hat.sqrt() + eps)                           # each push over its own size: about ±1 at step 1 (T4.1)
    return p - lr * step, m, v


def steps_in_rates(pushes, betas=(0.9, 0.999), eps=1e-8, corrected=True):
    """One number's move at each step, in learning rates (weight decay aside), for a list of its pushes."""
    b1, b2 = betas
    m = v = 0.0
    moves = []
    for t, g in enumerate(pushes, 1):
        m, v = b1 * m + (1 - b1) * g, b2 * v + (1 - b2) * g * g
        if corrected:
            moves.append(-(m / (1 - b1 ** t)) / (math.sqrt(v / (1 - b2 ** t)) + eps))
        else:
            moves.append(-m / (math.sqrt(v) + eps))               # without the corrections
    return moves


def decay_left(rates, weight_decay=0.01):
    """What weight decay alone leaves of a number: each step keeps 1 − weight_decay × that step's learning rate."""
    left = 1.0
    for lr in rates:
        left *= 1 - weight_decay * lr
    return left


def learning_rate(step, lr=3e-3, min_lr=3e-4, warmup=200, max_steps=100_000):
    """train.py's schedule (lr_at): a straight climb to lr over the warm-up, then down a cosine to min_lr at the last step."""
    if step < warmup:
        return lr * step / warmup
    done = min(1.0, (step - warmup) / (max_steps - warmup))
    return min_lr + 0.5 * (lr - min_lr) * (1 + math.cos(math.pi * done))


def first_steps(model, train_ids, steps, rate=learning_rate, B=64, T=32, watch=None):
    """The run's first steps, as train.py takes them (the seed set just before the model was made): each step's learning rate,
    64 random windows, their loss, the way back, then AdamW's step. Calls watch(model) after every step; returns its answers."""
    opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
    seen = []
    for step in range(steps):
        for group in opt.param_groups:
            group["lr"] = rate(step + 1)                          # this step's learning rate
        starts = torch.randint(len(train_ids) - T - 1, (B,))      # this step's 64 places
        x = torch.stack([train_ids[i:i + T] for i in starts])
        y = torch.stack([train_ids[i + 1:i + T + 1] for i in starts])
        loss = model(x, y)[1]
        opt.zero_grad(set_to_none=True)
        loss.backward()                                           # the way back
        opt.step()                                                # AdamW moves every number
        if watch:
            seen.append(watch(model))
    return seen


@torch.no_grad()
def check_loss(model, ids, starts, T=32):
    """The run's check: the average loss over fixed windows (check_starts), as train.py scores it every 500 steps."""
    model.eval()
    total = sum(model(torch.stack([ids[i:i + T] for i in s]), torch.stack([ids[i + 1:i + T + 1] for i in s]))[1].item() for s in starts)
    model.train()
    return total / len(starts)


# ---------------------------------------------------------------------------------------------------- T5 · the loop

def one_turn(model, opt, train_ids, step, B=64, T=32, clear=True):
    """One turn of train.py's loop, in its order: this step's learning rate, 64 random windows, their loss, the old gradients
    cleared, the way back, then AdamW's step. clear=False leaves the old gradients in (T5.1). Returns the turn's loss."""
    for group in opt.param_groups:
        group["lr"] = learning_rate(step + 1)                     # 1 · this step's learning rate
    starts = torch.randint(len(train_ids) - T - 1, (B,))          # 2 · 64 random places
    x = torch.stack([train_ids[i:i + T] for i in starts])
    y = torch.stack([train_ids[i + 1:i + T + 1] for i in starts])
    loss = model(x, y)[1]                                         # 3 · their loss
    if clear:
        opt.zero_grad(set_to_none=True)                           # 4 · the last turn's gradients cleared
    loss.backward()                                               # 5 · the way back adds each gradient to what .grad holds
    opt.step()                                                    # 6 · AdamW moves every number
    return loss.item()


def run_turns(model, train_ids, steps, check=None, every=50, clear=True, remake=False):
    """Turns 1 to `steps` from where the model is, scoring check(model) every `every` steps. remake=True makes a new AdamW every
    turn, so its averages and its step count are lost (T5.2). Returns [[step, check], ...]."""
    opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
    curve = []
    for step in range(steps + 1):
        if check and step % every == 0:
            curve.append([step, check(model)])
        if step == steps:
            return curve
        if remake:
            opt = torch.optim.AdamW(model.parameters(), lr=3e-3)  # a new AdamW: its averages start at 0 again
        one_turn(model, opt, train_ids, step, clear=clear)


def what_lasts(model, opt):
    """What carries on from one turn to the next: the learned numbers, AdamW's two averages for each, and its step count."""
    numbers = sum(p.numel() for p in model.parameters())
    averages = sum(s["exp_avg"].numel() + s["exp_avg_sq"].numel() for s in opt.state.values())
    steps = sorted({int(s["step"]) for s in opt.state.values()})
    return numbers, averages, steps


def saved(path):
    """What a saved file holds: each table's name and shape, how many numbers in all, and the file's size in bytes."""
    tables = torch.load(path, weights_only=True)
    shapes = {name: tuple(t.shape) for name, t in tables.items()}
    return shapes, sum(t.numel() for t in tables.values()), os.path.getsize(path)


# ---------------------------------------------------------------------------------------------------- T6 · the check

def other_checks(model, ids, seeds, batches=100, B=64, T=32):
    """The check on other windows: for each seed, its own 100 batches of 64 starts, drawn as train.py draws the check's, scored
    as check_loss scores them. The model's numbers never change: only the windows do."""
    scores = []
    for seed in seeds:
        g = torch.Generator().manual_seed(seed)                   # another seed: other windows
        starts = [torch.randint(len(ids) - T - 1, (B,), generator=g) for _ in range(batches)]
        scores.append(check_loss(model, ids, starts, T))
    return scores                                                 # one score per seed


def how_often_asked(starts, n, T=32):
    """How many characters of a text the check's windows ask about 0 times, once, twice, …: times_asked over those windows,
    counted by times. The first and the last characters are never asked: no window starts before the first or reaches the
    last."""
    asked = times_asked(starts, n, T)                             # [n]   one count per character
    return torch.bincount(asked).tolist()                         # [how many 0 times, once, twice, …]


def loss_by_position(model, ids, T=32):
    """The final score, position by position: answer_costs reads the whole text in windows of 32, side by side; each
    position's costs are averaged down the windows. Position 0 has one character to go on, position 31 all 32."""
    p, _ = answer_costs(model, ids, T)                            # [n·T]  every right answer's probability, window after window
    return (-p.log()).view(-1, T).mean(0)                         # [T]    one average cost per position


def keep_best(checks, margin=1e-4, patience=20):
    """train.py's rule, replayed over a log of checks ([step, held-back score] pairs): a check that beats the best so far by
    more than `margin` keeps a copy, and the wait starts again; any other check adds 1 to the wait; a wait of `patience`
    would stop the run. Returns the kept steps, the step it would stop at (None: never), and the kept copy's score."""
    best, wait, kept = float("inf"), 0, []
    for step, score in checks:
        if score < best - margin:
            best, wait = score, 0                                 # a new best: keep a copy
            kept.append(step)
        elif step:
            wait += 1                                             # no new best: one more check waited
        if patience and wait >= patience:
            return kept, step, best                               # train.py would stop the run here
    return kept, None, best

In [6]:
tok = Tokenizer(text)
cfg = Config(vocab_size=len(tok.chars))
ids = torch.tensor(tok.encode(text))                        # the whole text as ids
cut = int(0.9 * len(ids))
train_ids, val_ids = ids[:cut], ids[cut:]                   # the first 90% trains; the rest is held back
_, val_starts = check_starts(len(train_ids), len(val_ids))  # the run's held-back windows, the same every check

# step 0, in train.py's order: the seed, then the model (its starting numbers), then the first batch's 64 starts
torch.manual_seed(1337)
model0 = Prompter(cfg)
starts = torch.randint(len(train_ids) - 32 - 1, (64,))
x0 = torch.stack([train_ids[i:i + 32] for i in starts])        # [64×32]  the inputs
y0 = torch.stack([train_ids[i + 1:i + 33] for i in starts])    # [64×32]  their right answers
assert starts.tolist() == [160204, 793724, 810574, 964722, 40734, 423895, 818722, 483307, 510490, 928356, 614059, 30062, 737496, 407796, 59365, 998494, 864171, 929951, 942896, 637372, 408276, 289366, 811280, 165163, 748492, 428999, 260643, 560439, 371836, 390719, 776246, 678592, 338694, 316420, 711307, 461584, 554475, 251253, 546994, 358001, 64331, 976218, 589122, 206049, 243936, 44076, 936280, 819204, 162295, 566156, 935951, 895342, 457178, 537830, 274346, 823229, 14080, 899619, 528867, 495983, 90293, 96640, 21128, 975278]   # the recorded run's first batch
space, dollar, w = tok.chars.index(' '), tok.chars.index('$'), tok.chars.index('w')
print(f"the first batch: {tuple(x0.shape)} · {count_params(model0):,} learned numbers")
assert count_params(model0) == 9417

the first batch: (64, 32) · 9,417 learned numbers


# Section 1 · One step, number by number

AdamW's first step, number by number. Each chapter below follows its page: Picture it, Watch it in our run, then Read and run the
code.

## T4.1 · The first update

*How far does AdamW move each number at step 1?*

**Predict first** (the page asks it too, before its clip): at step 1 the learning rate is 0.000015. The space's readout bias has gradient −0.144; y's has
−0.00015, about 1,000 times smaller. How far does AdamW move each: both by 0.000015, the learning rate; the space's by
the rate times its gradient, and y's 1,000 times less; or only the space's, as y's gradient is too small to move it?

### T4.1.1 · Picture it

Every number now has a gradient. How should a gradient become a step? The plain way is gradient descent: each step is the learning
rate times the gradient. Picture two made-up gradients, 1 and 0.01. With a learning rate of 0.1, they step 0.1 and 0.001. One step
is 100 times smaller: a rate that suits one is wrong for the other. (The page's gradients are made up to show the idea.)

### T4.1.2 · Watch it in our run

*(The page's recorded clip, in words.)*

Here is step 1 of the run: every gradient, and the step AdamW, the run's optimizer, gave each. The learning rate is 0.000015.
Nearly every number, 9,256 of them, moved almost exactly that far. The space's bias, gradient
-0.144, moved up by the rate. So did the bias for y, its gradient about 1,000 times smaller. AdamW divides each number's
average push by its average size. At step 1 both are one gradient, so the step is the rate.

### T4.1.3 · Read and run the code

`adamw_step` in `code/training_steps.py` (copied in above) takes one step for any numbers. It keeps two averages for each number:
its pushes, with their signs, and the squares of their sizes. The step is the push average over the square root of the size
average, times the learning rate. The cell takes step 1 by hand for all 9,417 numbers and checks it against PyTorch's AdamW. At
step 1, a push over its own size is plus or minus 1: nearly every number moves the rate.

In [7]:
lr = learning_rate(1)                                       # step 1's learning rate
model0.zero_grad()
model0(x0, y0)[1].backward()                                # the first batch's gradients
before = [p.detach().clone() for p in model0.parameters()]
mine = [adamw_step(b, p.grad, torch.zeros_like(b), torch.zeros_like(b), 1, lr)[0] for b, p in zip(before, model0.parameters())]
torch.optim.AdamW(model0.parameters(), lr=lr).step()        # PyTorch's own step, on the same gradients
print('by hand against PyTorch, the largest difference:', f"{max((a - p.detach()).abs().max().item() for a, p in zip(mine, model0.parameters())):.1e}")
own = torch.cat([(b * (1 - lr * 0.01) - a).abs().ravel() for a, b in zip(mine, before)])   # each step, the pull toward 0 aside
moved_full = int(((own - lr).abs() <= 0.01 * lr).sum())
print(f"the learning rate: {lr:.6f} · moved by it, to within 1 in 100: {moved_full:,} of {own.numel():,}")
bias = dict(model0.named_parameters())['head_bias']
print(f"the space's bias: gradient {model0.head_bias.grad[space].item():.3f}, moved {bias[space].item():+.6f}")
print(f"y's bias:         gradient {model0.head_bias.grad[tok.chars.index('y')].item():.5f}, moved {bias[tok.chars.index('y')].item():+.6f}")
assert lr == 1.5e-05 and moved_full == 9256
assert max((a - p.detach()).abs().max().item() for a, p in zip(mine, model0.parameters())) < 1e-8   # float32 rounding
with torch.no_grad():                                       # put the numbers back, for the next cells
    for b, p in zip(before, model0.parameters()):
        p.copy_(b)
model0.zero_grad()

by hand against PyTorch, the largest difference: 3.7e-09
the learning rate: 0.000015 · moved by it, to within 1 in 100: 9,256 of 9,417
the space's bias: gradient -0.144, moved +0.000015
y's bias:         gradient -0.00015, moved +0.000015


### ✋ Your turn

A few numbers moved less than the rate: those whose gradient was tiny next to ε, AdamW's 0.00000001, added under the division.
How many moved less than 0.99 of the rate? Count them in `own`, and put the number in `answer`.

In [8]:
answer = None   # TODO: int((own < 0.99 * lr).sum())

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == int((own < 0.99 * lr).sum()), 'the numbers under 0.99 of the rate: (own < 0.99 * lr); how many: int(....sum())'
    print("✓ right:", answer)

Your turn: fill in `answer` above, then run this cell again.


In [9]:
#@title Show a solution { display-mode: "form" }
answer = int((own < 0.99 * lr).sum())
print(answer, 'numbers moved less than the rate at step 1')

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == int((own < 0.99 * lr).sum()), 'the numbers under 0.99 of the rate: (own < 0.99 * lr); how many: int(....sum())'
    print("✓ right:", answer)

161 numbers moved less than the rate at step 1
✓ right: 161


## T4.2 · Averages that start at 0

*Why does AdamW correct its averages?*

**Predict first** (the page asks it too, before its clip): AdamW's two averages start at 0. After step 1, m, the push average, holds 0.1 of the gradient, and v, the size average, 0.001
of its square. Without the corrections, how far would step 1 move a number: about 3.16 times the learning rate; a tenth of the learning
rate, as m is a tenth of the gradient; or exactly the learning rate, as the two shrink alike?

### T4.2.1 · Picture it

AdamW's averages start at 0. What does that get wrong? Picture made-up pushes of 2. Each step, the average keeps 0.9 of itself
and adds 0.1 of the push. It reads 0.2, then 0.38, then 0.542: far below 2. Divide by 1 minus 0.9 to the power of the step, and it
reads 2 from the start: the correction. (The page's pushes are made up to show the idea.)

### T4.2.2 · Watch it in our run

*(The page's recorded clip, in words.)*

Here are the run's first 1,000 steps: how much bigger AdamW's steps would be without the corrections. At step 1,
3.16 times. The size average lags further behind than the push average, so the step comes out too big. It peaks at
step 12, at 6.6 times, and is still 2.35 times at step 200. The push
average's correction fades by step 44. The size average's lasts: at step 1,000, still 1.26 times.

### T4.2.3 · Read and run the code

`steps_in_rates` follows one number's two averages, push by push, and gives each step's move in learning rates. With the corrections,
it divides each average by 1 minus what it keeps, to the power of the step. Fed one push of 0.3 without them, it moves 3.16 learning rates. With them,
1.00. So with the corrections, the first step is the rate for nearly every push.

In [10]:
print('one push of 0.3, uncorrected:', [round(m, 2) for m in steps_in_rates([0.3], corrected=False)], 'learning rates')
print('one push of 0.3, corrected:  ', [round(m, 2) for m in steps_in_rates([0.3])])
ups = [-m for m in steps_in_rates([1.0] * 1000, corrected=False)]   # the same push every step, uncorrected
peak = max(ups)
print(f"uncorrected: {ups[0]:.2f} times at step 1 · {peak:.1f} at step {ups.index(peak) + 1} · {ups[199]:.2f} at 200 · {ups[999]:.2f} at 1,000")
assert abs(ups[0] - 3.16228) < 1e-5 and ups.index(peak) + 1 == 12
assert abs(peak - 6.56852) < 1e-4 and abs(ups[199] - 2.34823) < 1e-4 and abs(ups[999] - 1.25758) < 1e-4

one push of 0.3, uncorrected: [-3.16] learning rates
one push of 0.3, corrected:   [-1.0]
uncorrected: 3.16 times at step 1 · 6.6 at step 12 · 2.35 at 200 · 1.26 at 1,000


### ✋ Your turn

The push average's correction divides by 1 − 0.9 to the power of the step. At which step does that first reach 0.99, so the correction all but
fades? Find the first `t` with `1 - 0.9 ** t >= 0.99`, and put it in `answer`.

In [11]:
answer = None   # TODO: next(t for t in range(1, 1000) if 1 - 0.9 ** t >= 0.99)

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == next(t for t in range(1, 1000) if 1 - 0.9 ** t >= 0.99), 'count up from 1: next(t for t in range(1, 1000) if 1 - 0.9 ** t >= 0.99)'
    print("✓ right:", answer)

Your turn: fill in `answer` above, then run this cell again.


In [12]:
#@title Show a solution { display-mode: "form" }
answer = next(t for t in range(1, 1000) if 1 - 0.9 ** t >= 0.99)
print(f'step {answer}: and the size average, 1 − 0.999 to the power of the step, there: {1 - 0.999 ** answer:.3f}')

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == next(t for t in range(1, 1000) if 1 - 0.9 ** t >= 0.99), 'count up from 1: next(t for t in range(1, 1000) if 1 - 0.9 ** t >= 0.99)'
    print("✓ right:", answer)

step 44: and the size average, 1 − 0.999 to the power of the step, there: 0.043
✓ right: 44


# Section 2 · Many steps

AdamW over many steps: what its averages remember, the pull toward 0, and the learning rate's curve.

## T4.3 · What AdamW remembers

*Which numbers move far, once the averages hold many steps?*

**Predict first** (the page asks it too, before its clip): $ is the right answer once in the whole training text. w is about 1 in 64 of its characters, about what step 0's even guess gives
every character. By step 500,
whose readout bias has moved further from where it started: $'s, as its pushes point the same way; w's, as a common letter gets far
more pushes; or about the same, as AdamW steps every number by the learning rate?

### T4.3.1 · Picture it

One batch's gradient is noisy: the next batch may push the other way. What should a step follow? AdamW's two averages remember many
steps: the pushes, with their signs, and their sizes. Picture made-up pushes that agree, +1, +1, +1: full steps, one way. Pushes that
flip, +1, −1, +1, cancel in the push average, but not in the size: the steps shrink. (The page's pushes are made up to show the
idea.)

### T4.3.2 · Watch it in our run

*(The page's recorded clip, in words.)*

Here are the run's first 500 steps, and two readout biases: for $, and for w. $ is almost never right, so its push pointed down at
497 of 500 steps. Its bias fell 0.15. The push for w changed sign 233 times. It kept
cancelling, and ended 0.003 from its start. Stepping the full rate one way would move a number
1.2. $'s pushes shrank as its probability fell, and its steps shrank with them.

### T4.3.3 · Read and run the code

`first_steps` takes `train.py`'s steps, one after another, from the seed. Each step sets its learning rate, draws 64 windows, runs
the way back, then lets AdamW move every number; a `watch` reads whatever you like after every step. The cell replays 500 steps; its watch
reads the two biases after every step: they land exactly where the run put them. Fed to `steps_in_rates`,
pushes that flip for 1,000 steps end at 0.05 of a rate.

In [13]:
torch.manual_seed(1337)                                   # train.py's order: the seed, then the model, then the steps
model = Prompter(cfg)
start = torch.cat([p.detach().ravel().clone() for p in model.parameters()])
seen = first_steps(model, train_ids, 500, watch=lambda m: (m.head_bias[dollar].item(), m.head_bias[w].item()))
moved = (torch.cat([p.detach().ravel() for p in model.parameters()]) - start).abs()
print(f"after 500 steps: the bias for $ {seen[-1][0]:+.2f} · for w {seen[-1][1]:+.3f}")
print(f"half of all {moved.numel():,} numbers moved less than {moved.median().item():.3f} · the full rate one way: {sum(learning_rate(s) for s in range(1, 501)):.1f}")
flips = steps_in_rates([1.0 if t % 2 else -1.0 for t in range(1, 1001)])
print(f"pushes that flip, after 1,000 steps: {abs(flips[-1]):.2f} of a rate")
assert abs(seen[-1][0] - -0.152667) < 1e-5 and abs(seen[-1][1] - -0.00288261) < 1e-5
assert abs(moved.median().item() - 0.0517141) < 1e-5 and abs(abs(flips[-1]) - 0.0526316) < 1e-5
print(f"the held-back check at step 500: {check_loss(model, val_ids, val_starts):.4f} (the run's log: 2.3036)")

after 500 steps: the bias for $ -0.15 · for w -0.003
half of all 9,417 numbers moved less than 0.052 · the full rate one way: 1.2
pushes that flip, after 1,000 steps: 0.05 of a rate


the held-back check at step 500: 2.3036 (the run's log: 2.3036)


### ✋ Your turn

w's bias wandered before it settled. Find the lowest it went in the 500 steps: the smallest second value in `seen`. Put it in
`answer`.

In [14]:
answer = None   # TODO: min(s[1] for s in seen)

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == min(s[1] for s in seen), 'the second value of each step: s[1] for s in seen; the smallest: min(...)'
    print("✓ right:", answer)

Your turn: fill in `answer` above, then run this cell again.


In [15]:
#@title Show a solution { display-mode: "form" }
answer = min(s[1] for s in seen)
print(f"w's bias went as low as {answer:.4f}, at step {[s[1] for s in seen].index(answer) + 1}")

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == min(s[1] for s in seen), 'the second value of each step: s[1] for s in seen; the smallest: min(...)'
    print("✓ right:", answer)

w's bias went as low as -0.0257, at step 206
✓ right: -0.025670809671282768


## T4.4 · The W in AdamW

*What does weight decay do to a number?*

**Predict first** (the page asks it too, before its clip): every step, weight decay keeps 1 − 0.01 × the learning rate of every number: at most 0.00003 of it taken off.
With no push at all, how much of a number would be left after all 100,000 steps: about a fifth; almost all of it, as the pull is
tiny; or none, as it would reach 0?

### T4.4.1 · Picture it

AdamW's steps follow the pushes. Its W adds one thing that does not: weight decay, a pull toward 0. Picture a made-up number at 2,
and no push at all. Each step keeps 0.9 of it: 2, then 1.8, then 1.62. The pull is a share: big numbers lose more, and 0 stays 0.
(The page's number and its pull are made up to show the idea.)

### T4.4.2 · Watch it in our run

*(The page's recorded clip, in words.)*

Here is weight decay over the whole run, and the 84 LayerNorm scales, every one starting at 1. Weight decay alone would
leave 0.19 of a number by the run's end: about a fifth. Yet the kept scales average 0.75 in the first
LayerNorm and 1.46 in the second. Before the readout, 2.19. Their pushes held every one above
0.36. A number stays big only while its pushes keep asking.

### T4.4.3 · Read and run the code

`decay_left` works out what the pull alone leaves of a number. Each step keeps 1 minus 0.01 times that step's learning rate; it
multiplies those shares together. Over the run's 100,000 learning rates, it leaves 0.192. In `train.py`, AdamW
gets every number in one group: biases and scales are pulled too. The cell also reads the kept LayerNorm scales from the run's kept
numbers.

In [16]:
rates = [learning_rate(s) for s in range(1, 100000 + 1)]   # the run's learning rate at every step
print(f"weight decay alone over the run leaves {decay_left(rates):.3f} of a number · the most one step takes off: {0.01 * max(rates):.5f}")
opt = torch.optim.AdamW(model0.parameters(), lr=3e-3)       # as train.py makes it
print(f"train.py's AdamW: {len(opt.param_groups)} group · {sum(p.numel() for p in opt.param_groups[0]['params']):,} numbers · weight_decay {opt.param_groups[0]['weight_decay']}")
assert abs(decay_left(rates) - 0.192104) < 1e-6 and len(opt.param_groups) == 1

# the kept numbers: the run's best check, step 98,000 (the course's prompter.json, the same numbers as best.pt)
paths = [os.path.join(here, '..', 'model', 'prompter.json'),                                                     # in the labs
         os.path.join(here, '..', '..', '..', '..', 'src', 'specimen', 'tiny-transformer', 'data', 'prompter.json')]   # in the course
found = [p for p in paths if os.path.exists(p)]
assert found, "the model's trained numbers are not here: run the Setup cell at the top"
exp = json.load(open(found[0]))
state = {t['name']: torch.tensor(np.frombuffer(base64.b64decode(t['data']), dtype='<f4').reshape(t['shape']).copy())
         for t in exp['weights']['tensors']}
scales = [state[k] for k in ('blocks.0.ln1.weight', 'blocks.0.ln2.weight', 'ln_f.weight')]   # every one 1 at the start
for name, g in zip(('first LayerNorm', 'second LayerNorm', 'before the readout'), scales):
    print(f"{name:>18}: average {g.mean().item():.2f}, smallest {g.min().item():.2f}")
assert exp['checkpoint_step'] == 98000 and abs(min(g.min().item() for g in scales) - 0.357219) < 1e-4

weight decay alone over the run leaves 0.192 of a number · the most one step takes off: 0.00003
train.py's AdamW: 1 group · 9,417 numbers · weight_decay 0.01
   first LayerNorm: average 0.75, smallest 0.56
  second LayerNorm: average 1.46, smallest 1.06
before the readout: average 2.19, smallest 0.36


### ✋ Your turn

The pull is a share: what would it leave over the run if the rate stayed at its peak, 0.003, every step? Use `decay_left` on
100,000 rates of 0.003, and put the result in `answer`. Is it more or less than the run's 0.19?

In [17]:
answer = None   # TODO: decay_left([0.003] * 100000)

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert abs(answer - decay_left([0.003] * 100000)) < 1e-12, 'the same rate every step: decay_left([0.003] * 100000)'
    print("✓ right:", answer)

Your turn: fill in `answer` above, then run this cell again.


In [18]:
#@title Show a solution { display-mode: "form" }
answer = decay_left([0.003] * 100000)
print(f"at the peak rate throughout: {answer:.3f} left, against the run's 0.19")

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert abs(answer - decay_left([0.003] * 100000)) < 1e-12, 'the same rate every step: decay_left([0.003] * 100000)'
    print("✓ right:", answer)

at the peak rate throughout: 0.050 left, against the run's 0.19
✓ right: 0.04978482795551448


## T4.5 · The learning rate's curve

*What does the learning rate do over the run?*

**Predict first** (the page asks it too, before its clip): the run's learning rate climbs from 0 to 0.003 over its first 200 steps, the warm-up. Replayed from the same seed with no
warm-up, 0.003 from step 1, how would it score at step 1,000: about the same; far worse, as big first steps throw the model
off; or far better, as the warm-up wastes 200 steps?

### T4.5.1 · Picture it

With AdamW, the learning rate sets about how far a number steps. How big should the steps be? Picture a made-up number, stepping
down a valley. Big steps cover ground fast, then hop back and forth across the bottom. Small steps settle, but crawl. So the run
starts big and ends small. First comes a short climb from 0, the warm-up, while the averages are young. (The page's valley is made up to show the idea.)

### T4.5.2 · Watch it in our run

*(The page's recorded clip, in words.)*

Here is the run's learning rate, over all 100,000 steps. It climbs to 0.003 over the first 200 steps. Then it falls along a
smooth curve, a cosine, to 0.0003. Replayed with no warm-up, the loss falls faster: 2.68 against
3.20 at step 100. By step 1,000 the two meet: 2.121 and 2.124. In this replay, our small model did not
need a warm-up.

### T4.5.3 · Read and run the code

The function `learning_rate` gives `train.py`'s rate for any step. Below step 200 it climbs in a straight line. After that, a cosine
carries it down. The cell runs `first_steps` twice from the seed: with the warm-up, and without it. Then `check_loss` scores both on
the held-back windows, as the run's own check does.

In [19]:
print(f"the learning rate: step 1 {learning_rate(1):.6f} · step 200 {learning_rate(200)} · step 100,000 {learning_rate(100000):.4f}")
scores = {}
for name, rate in (('with the warm-up', learning_rate), ('no warm-up', lambda step: 0.003)):
    torch.manual_seed(1337)                               # the same seed, so the same starting numbers and batches
    model, done = Prompter(cfg), [0]
    def at_checks(m):                                       # after every step: the held-back check at steps 100 and 1,000
        done[0] += 1
        return check_loss(m, val_ids, val_starts) if done[0] in (100, 1000) else None
    scores[name] = [v for v in first_steps(model, train_ids, 1000, rate=rate, watch=at_checks) if v is not None]
    print(f"{name:>17}: the held-back check at step 100 {scores[name][0]:.2f} · at step 1,000 {scores[name][1]:.3f}")
assert abs(scores['with the warm-up'][1] - 2.12428) < 1e-4 and abs(scores['no warm-up'][1] - 2.12064) < 1e-4
assert abs(scores['with the warm-up'][0] - 3.2038) < 1e-4 and abs(scores['no warm-up'][0] - 2.67967) < 1e-4

the learning rate: step 1 0.000015 · step 200 0.003 · step 100,000 0.0003


 with the warm-up: the held-back check at step 100 3.20 · at step 1,000 2.124


       no warm-up: the held-back check at step 100 2.68 · at step 1,000 2.121


### ✋ Your turn

Where is the learning rate halfway down its cosine? Find the step where `learning_rate` first falls below the middle of
0.003 and 0.0003, and put it in `answer`. (Hint: a cosine is halfway at the middle of its run.)

In [20]:
answer = None   # TODO: next(s for s in range(200, 100000 + 1) if learning_rate(s) < (0.003 + 0.0003) / 2)

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == next(s for s in range(200, 100000 + 1) if learning_rate(s) < (0.003 + 0.0003) / 2), 'step up from 200 until learning_rate(s) is under the middle of the two rates'
    print("✓ right:", answer)

Your turn: fill in `answer` above, then run this cell again.


In [21]:
#@title Show a solution { display-mode: "form" }
answer = next(s for s in range(200, 100000 + 1) if learning_rate(s) < (0.003 + 0.0003) / 2)
print(f'halfway down at step {answer:,}: the middle of steps 200 and 100,000')

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == next(s for s in range(200, 100000 + 1) if learning_rate(s) < (0.003 + 0.0003) / 2), 'step up from 200 until learning_rate(s) is under the middle of the two rates'
    print("✓ right:", answer)

halfway down at step 50,101: the middle of steps 200 and 100,000
✓ right: 50101


AdamW divides each number's average push by its average size, so at step 1 nearly every number, 9,256 of
9,417, moved the learning rate, 0.000015, whatever its gradient. Its averages start at 0, so it corrects them; uncorrected,
step 1 would be 3.16 times too big. Over many steps, steady pushes that agree travel and pushes that flip cancel: the
bias for $ fell 0.15 in 500 steps, w's ended 0.003 away. Weight decay pulls every
number toward 0, 0.19 left over the run on its own; and the learning rate climbs to 0.003, then falls to
0.0003.

Next: **T5 · the loop**, which repeats that step 100,000 times.